In [ ]:
import tqdm

In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

In [ ]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [ ]:
len(titles_with_tags)

In [ ]:
data = []

In [ ]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [ ]:
data[0]

In [ ]:
data[-1]

In [ ]:
df = pd.DataFrame(data=data, columns=["title_rus", "tag"])

In [ ]:
df

In [ ]:
dfgr = df.groupby(by=["tag"]).agg({"title_rus": "nunique"}).reset_index()
dfgr.columns = ["tag", "count"]
dfgr = dfgr.sort_values(by=["count"], ascending=False).reset_index(drop=True)
dfgr.head(32)

In [ ]:
selected_tags = dfgr.head(3)["tag"].to_list()

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create several possible user profiles with own preferences in provided field of science.
Return stricly JSON output. Be brief.
For example.
Input data:
```json
{"machine_learning": ["Исследование методов решения задачи линейной регрессии", "Исследование методов решения задачи линейной классификации", "Исследование методов решения задачи обучения с подкреплением"]}
```
Your output:
```json
{"machine_learning_researcher_classic_algorithms": "A user, which is interested in machine learning and classic algorithms in particular"}
{"machine_learning_researcher_reinforcement_learning": "A user, which is interested in machine learning and reinforcement learning in particular"}
```
"""

In [ ]:
messages = []

In [ ]:
for tag in selected_tags:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        tag: list(df[df["tag"] == tag]["title_rus"].unique())
                    }, ensure_ascii=False
                ),
            },
        )
    )

In [ ]:
messages[0][1]["content"]

In [ ]:
len(messages[0][1]["content"])

In [ ]:
answers = {}

In [ ]:
for tag, message in tqdm.tqdm(list(zip(selected_tags, messages))):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=2048)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_str = response

    answers.update(json.loads(json_str))

In [ ]:
len(answers)

In [ ]:
answers

In [ ]:
# with open("artificial_profiles.json", "w") as f:
#     json.dump(answers, f)